# Xovato AI Studio — GPU Colab

One notebook for **every model feature** in `Xovato-cpu-ai-studio`, running on a Colab **GPU**.

The backend code is used **unchanged** — this notebook imports the real `app/` modules
(`image_processing`, `face_restoration`, `audio_processing`, `voice_engine`, `shorts_pipeline`, …)
and then patches the device layer so the same functions run on CUDA instead of CPU.

### Features in the picker
| Feature | Model / engine |
| --- | --- |
| Text → Speech | OmniVoice (`k2-fsa/OmniVoice`), fp16 on GPU |
| Voice Cloning | OmniVoice voice-clone prompt from your reference clip |
| Speech → Text (+ SRT) | faster-whisper, float16 on GPU |
| Image Enhancement | `app.image_processing` analysis-driven corrections |
| Background Removal | U²-Net-p ONNX, CUDA execution provider |
| Face Restoration | YuNet + GPEN-BFR-256 ONNX, CUDA execution provider |
| Audio Enhancement | `app.audio_processing` denoise + LUFS + limiter |
| Shorts / vertical clips | whisper → moment ranking → ffmpeg NVENC render |
| REST API server | the real FastAPI app on a public Colab URL |

### How to run
1. **Runtime → Change runtime type → T4 GPU** (or better), then Save.
2. Run cells ① → ⑤ **in order, once**.
3. Use the **Feature Studio** in cell ⑤: pick a feature, set options, press **Run**.

Cells ① – ④ are setup. Cell ⑤ is the UI you actually work in — rerun it as often as you like.

In [ ]:
#@title ① Runtime check — GPU, CPU, RAM, disk { display-mode: "form" }
import os, shutil, subprocess, sys, platform

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120).stdout.strip()
    except Exception as error:
        return f"(failed: {error})"

print("Python     :", platform.python_version())
print("Platform   :", platform.platform())
print("CPU cores  :", os.cpu_count())

mem = sh("cat /proc/meminfo | head -1")
print("RAM        :", mem or "unknown")

total, used, free = shutil.disk_usage("/content")
print(f"Disk       : {free/1e9:.1f} GB free of {total/1e9:.1f} GB")

print("ffmpeg     :", shutil.which("ffmpeg") or "NOT FOUND")

print("\n--- GPU ---")
smi = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
if smi and "failed" not in smi:
    print("GPU        :", smi)
else:
    print("GPU        : NONE DETECTED")
    print("\n  >>> Runtime -> Change runtime type -> Hardware accelerator: GPU (T4 or better), then Save")
    print("  >>> Everything below still runs on CPU, just far slower.")

try:
    import torch
    print("torch      :", torch.__version__, "| cuda available:", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
except ImportError:
    print("torch      : not installed yet (cell ② installs what is missing)")

In [ ]:
#@title ② Install dependencies and fetch the app code { display-mode: "form" }
#@markdown Leave a box unticked to skip that install and save a few minutes.
REPO_URL   = "https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git" #@param {type:"string"}
BRANCH     = "main" #@param {type:"string"}
APP_DIR    = "/content/Xovato-cpu-ai-studio" #@param {type:"string"}
INSTALL_TTS        = True  #@param {type:"boolean"}
INSTALL_WHISPER    = True  #@param {type:"boolean"}
INSTALL_ONNX_GPU   = True  #@param {type:"boolean"}

import os, subprocess, sys, textwrap
from pathlib import Path

def run(cmd, what):
    print(f"  {what} ...", end=" ", flush=True)
    done = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if done.returncode == 0:
        print("ok")
    else:
        print("FAILED")
        print(textwrap.indent((done.stderr or done.stdout or "")[-2500:], "    "))
    return done.returncode == 0

def pip(spec, what=None):
    return run(f"{sys.executable} -m pip install -q {spec}", what or f"pip install {spec}")

# --- 1. app code ------------------------------------------------------------
print("Source code")
app_dir = Path(APP_DIR)
if (app_dir / ".git").exists():
    run(f"git -C {app_dir} fetch --depth 1 origin {BRANCH} && git -C {app_dir} checkout -q {BRANCH} && git -C {app_dir} reset -q --hard origin/{BRANCH}", "updating existing clone")
elif (app_dir / "app").exists():
    print("  using the existing folder (not a git clone) ... ok")
else:
    run(f"git clone -q --depth 1 --branch {BRANCH} {REPO_URL} {app_dir}", f"cloning {REPO_URL}")

if not (app_dir / "app" / "config.py").exists():
    raise SystemExit(f"App code was not found at {app_dir}. Check REPO_URL / BRANCH above, "
                     f"or upload the project folder to {app_dir}.")

if str(app_dir) not in sys.path:
    sys.path.insert(0, str(app_dir))
os.chdir(app_dir)

# --- 1b. import shim --------------------------------------------------------
# The repo ships BOTH `app/services.py` (the real JobStore / VoiceStore / circuit
# breakers) and an `app/services/` facade package. Python resolves a package
# before a module of the same name, so `from app.services import job_store`
# picks the empty facade and raises ImportError — in this notebook, and in
# `app/main.py` and `app/shorts_service.py` just the same. Loading the file
# explicitly under the name `app.services` fixes every later import, because
# sys.modules is consulted before the filesystem.
BOOTSTRAP = Path("/content/xovato_bootstrap.py")
BOOTSTRAP.write_text('''"""Make `app.services` resolve to app/services.py, not the stub package."""

import importlib
import importlib.util
import sys
from pathlib import Path

APP_DIR = Path("/content/Xovato-cpu-ai-studio")


def _load_by_path(name, file, search_locations=None):
    spec = importlib.util.spec_from_file_location(
        name, file, submodule_search_locations=search_locations
    )
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module          # register before exec, for circular imports
    spec.loader.exec_module(module)
    return module


def install(app_dir=APP_DIR):
    app_dir = Path(app_dir)
    package = app_dir / "app"
    services_file = package / "services.py"

    if not package.is_dir():
        raise SystemExit(f"No `app` package at {package} - run cell 2 first (it clones the repo).")
    if not (package / "__init__.py").exists():
        raise SystemExit(f"{package / '__init__.py'} is missing - the clone is incomplete. Re-run cell 2.")

    if str(app_dir) not in sys.path:
        sys.path.insert(0, str(app_dir))

    # The repo is normally cloned *after* the kernel started, so the import
    # system's cached directory listings can still insist `app` does not exist.
    importlib.invalidate_caches()

    try:
        import app
    except ModuleNotFoundError:
        app = _load_by_path("app", package / "__init__.py", [str(package)])

    if not services_file.exists():
        return None  # layout fixed upstream; nothing to shim

    loaded = sys.modules.get("app.services")
    if loaded is not None and hasattr(loaded, "job_store"):
        return loaded  # already installed

    module = _load_by_path("app.services", services_file)
    app.services = module
    return module
''', encoding="utf-8")

print("  app.services shim written")

# --- 2. python packages -----------------------------------------------------
print("\nCore packages")
pip("pillow soundfile pydantic-settings 'fastapi>=0.115' 'uvicorn[standard]' python-multipart httpx psutil",
    "core runtime")
try:
    import ipywidgets  # noqa: F401
    print("  ipywidgets ... already present")
except ImportError:
    pip("ipywidgets", "ipywidgets")

if INSTALL_ONNX_GPU:
    print("\nONNX Runtime (GPU)")
    # The CPU and GPU wheels cannot coexist: onnxruntime-gpu must own the package name.
    run(f"{sys.executable} -m pip uninstall -y -q onnxruntime", "removing CPU onnxruntime")
    if not pip("onnxruntime-gpu", "onnxruntime-gpu"):
        pip("onnxruntime", "falling back to CPU onnxruntime")

if INSTALL_WHISPER:
    print("\nTranscription")
    pip("faster-whisper", "faster-whisper (CTranslate2, CUDA capable)")

if INSTALL_TTS:
    print("\nText to speech")
    # `omnivoice` pulls transformers>=5.3 and the audio stack; torch already
    # ships with Colab and is left untouched so the CUDA build stays intact.
    pip("omnivoice", "omnivoice (k2-fsa)")
    pip("'huggingface_hub[hf_transfer]'", "huggingface_hub")

# --- 3. activate the import shim -------------------------------------------
# Only now, because loading app/services.py pulls in app.config -> pydantic_settings.
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import xovato_bootstrap  # noqa: E402

services = xovato_bootstrap.install(app_dir)
print()
print("app.services shim:", "active" if services else "not needed (layout already fixed)")

# --- 4. environment ---------------------------------------------------------
os.environ["ENHANCEAI_USE_REAL_OMNIVOICE"] = "true"      # real model, not the preview tone
os.environ["ENHANCEAI_OMNIVOICE_QUANTIZE_INT8"] = "false" # int8 is a CPU trick; it hurts on GPU
os.environ["HF_HOME"] = "/content/hf"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["SHORTS_TRANSCRIBE_PROVIDER"] = "local"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

OUTPUT_DIR = Path("/content/xovato_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nApp dir     : {app_dir}")
print(f"Outputs dir : {OUTPUT_DIR}")
print("\nDone. Run cell ③ next.")

In [ ]:
#@title ③ Download the models { display-mode: "form" }
#@markdown The repo ships code only — weights are gitignored. Untick anything you will not use.
GET_SEGMENTATION = True  #@param {type:"boolean"}
GET_FACE         = True  #@param {type:"boolean"}
GET_OMNIVOICE    = True  #@param {type:"boolean"}
#@markdown `k2-fsa/OmniVoice` is ~2.5 GB. Whisper weights download themselves on first use.
WHISPER_SIZE = "base" #@param ["tiny", "base", "small", "medium", "large-v3"]

import os, sys, time, urllib.request
from pathlib import Path

APP_DIR = Path("/content/Xovato-cpu-ai-studio")
sys.path.insert(0, "/content")
import xovato_bootstrap  # app.services shim written by cell 2; see its docstring

xovato_bootstrap.install(APP_DIR)
os.environ["SHORTS_WHISPER_MODEL"] = WHISPER_SIZE

# hf_transfer makes the 2.5 GB checkpoint download far faster, but the env flag
# raises if the package is absent, so only ask for it when it is actually there.
try:
    import hf_transfer  # noqa: F401
except ImportError:
    os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

MODELS = APP_DIR / "models"

# Each entry lists mirrors; the first one that yields a plausible file wins.
DOWNLOADS = [
    ("u2netp.onnx (background removal)", MODELS / "segmentation" / "u2netp.onnx", 3_000_000, GET_SEGMENTATION, [
        "https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2netp.onnx",
    ]),
    ("yunet.onnx (face detection)", MODELS / "face" / "yunet.onnx", 100_000, GET_FACE, [
        "https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    ]),
    ("gpen_bfr_256.onnx (face restoration)", MODELS / "face" / "gpen_bfr_256.onnx", 20_000_000, GET_FACE, [
        "https://github.com/facefusion/facefusion-assets/releases/download/models-3.0.0/gpen_bfr_256.onnx",
    ]),
]

def fetch(label, destination, min_bytes, urls):
    if destination.exists() and destination.stat().st_size >= min_bytes:
        print(f"  {label}: already present ({destination.stat().st_size/1e6:.1f} MB)")
        return True

    destination.parent.mkdir(parents=True, exist_ok=True)
    for url in urls:
        partial = destination.with_suffix(destination.suffix + ".part")
        try:
            print(f"  {label}: downloading ...", end=" ", flush=True)
            started = time.time()
            urllib.request.urlretrieve(url, partial)
            size = partial.stat().st_size
            # A Git-LFS pointer or an error page is a few hundred bytes, not a model.
            if size < min_bytes:
                partial.unlink(missing_ok=True)
                print(f"only {size} bytes — that is not the model, trying the next mirror")
                continue
            partial.replace(destination)
            print(f"ok ({size/1e6:.1f} MB in {time.time()-started:.0f}s)")
            return True
        except Exception as error:
            partial.unlink(missing_ok=True)
            print(f"failed ({error})")
    print(f"  {label}: NOT AVAILABLE — the feature that needs it will say so when you run it.")
    return False

print("ONNX models")
for label, destination, min_bytes, wanted, urls in DOWNLOADS:
    if wanted:
        fetch(label, destination, min_bytes, urls)
    else:
        print(f"  {label}: skipped")

if GET_OMNIVOICE:
    print("\nOmniVoice TTS (k2-fsa/OmniVoice, ~2.5 GB)")
    target = MODELS / "omnivoice" / "k2-fsa_OmniVoice"
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id="k2-fsa/OmniVoice",
            local_dir=str(target),
            allow_patterns=["*.json", "*.safetensors", "*.jinja", "*.txt", "*.model", "audio_tokenizer/*"],
        )
        from app.omnivoice_runtime import model_ready
        print("  complete:", "yes" if model_ready(target) else "NO — some files are missing")
    except Exception as error:
        print(f"  failed: {error}")
        print("  Text to speech and voice cloning will not run until this succeeds.")
else:
    print("\nOmniVoice TTS: skipped")

print("\n--- Model status as the app sees it ---")
from app.model_manager import model_status
for row in model_status():
    print(f"  {'PRESENT' if row['exists'] else 'missing':>8}  {row['id']:<20} {row['feature']}")
print(f"\n  faster-whisper '{WHISPER_SIZE}' downloads on first transcription.")

In [ ]:
#@title ④ Switch the app onto the GPU { display-mode: "form" }
#@markdown The backend was written CPU-only. Nothing in `app/` is edited — these patches
#@markdown swap the device layer underneath it: ONNX Runtime execution providers,
#@markdown OmniVoice load precision, faster-whisper device, and the ffmpeg video encoder.

import os, subprocess, sys, time
from pathlib import Path

APP_DIR = Path("/content/Xovato-cpu-ai-studio")
sys.path.insert(0, "/content")
import xovato_bootstrap  # app.services shim written by cell 2; see its docstring

xovato_bootstrap.install(APP_DIR)
os.chdir(APP_DIR)

import torch
CUDA = torch.cuda.is_available()
DEVICE = "cuda:0" if CUDA else "cpu"
report = []

if CUDA:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    report.append(f"torch      -> {torch.cuda.get_device_name(0)} (TF32 on)")
else:
    torch.set_num_threads(max(1, os.cpu_count() or 1))
    report.append("torch      -> CPU (no GPU runtime attached)")

# --------------------------------------------------------------------------- #
# 1. ONNX Runtime — background removal, face detection, face restoration.
#    app/image_processing.py and app/face_restoration.py both hardcode
#    providers=["CPUExecutionProvider"]. They import onnxruntime *inside* the
#    session builders, so replacing the attribute here reaches both.
# --------------------------------------------------------------------------- #
ORT_PROVIDERS = ["CPUExecutionProvider"]
try:
    import onnxruntime as ort

    available = ort.get_available_providers()
    preferred = [p for p in ("CUDAExecutionProvider",) if p in available]
    ORT_PROVIDERS = preferred + ["CPUExecutionProvider"]

    _OrigSession = ort.InferenceSession

    def _gpu_session(path_or_bytes, sess_options=None, providers=None, provider_options=None, **kwargs):
        """Ignore the caller's CPU-only provider list and try CUDA first."""
        try:
            return _OrigSession(path_or_bytes, sess_options=sess_options,
                                providers=ORT_PROVIDERS, provider_options=provider_options, **kwargs)
        except Exception as error:
            # A missing CUDA EP should degrade to CPU, not break the feature.
            print(f"  [onnxruntime] CUDA provider unavailable ({error}); using CPU.")
            return _OrigSession(path_or_bytes, sess_options=sess_options,
                                providers=["CPUExecutionProvider"], provider_options=provider_options, **kwargs)

    ort.InferenceSession = _gpu_session
    report.append("onnxruntime-> " + " + ".join(ORT_PROVIDERS))
except ImportError:
    report.append("onnxruntime-> NOT INSTALLED (background removal / face models disabled)")

# --------------------------------------------------------------------------- #
# 2. OmniVoice — app/voice_engine.py loads float32 on CPU and applies int8
#    dynamic quantisation, which is a CPU-only optimisation. On GPU the win is
#    fp16 weights instead, and the vendored-source check does not apply because
#    omnivoice is installed as a pip package here.
# --------------------------------------------------------------------------- #
from app.voice_engine import VoiceEngine, VoiceEngineError, voice_engine

def _gpu_load(self, settings):
    if self._model is not None:
        return self._model

    with self._model_lock:
        if self._model is not None:
            return self._model

        model_dir = settings.omnivoice_model_dir
        if not model_dir.exists():
            raise VoiceEngineError(
                f"OmniVoice weights are not at {model_dir}. Rerun cell 3 with GET_OMNIVOICE ticked."
            )

        try:
            from omnivoice.models.omnivoice import OmniVoice
        except ImportError:
            try:
                from omnivoice import OmniVoice
            except ImportError as error:
                raise VoiceEngineError("The `omnivoice` package is not installed; rerun cell 2.") from error

        started = time.monotonic()
        dtype = torch.float16 if CUDA else torch.float32
        try:
            try:
                model = OmniVoice.from_pretrained(str(model_dir), device_map=DEVICE, dtype=dtype)
            except TypeError:
                # Older releases used the transformers spelling.
                model = OmniVoice.from_pretrained(str(model_dir), device_map=DEVICE, torch_dtype=dtype)
            model.eval()
        except Exception as error:
            raise VoiceEngineError(f"Could not load OmniVoice on {DEVICE}: {error}") from error

        self._load_seconds = time.monotonic() - started
        self._model = model
        print(f"  [omnivoice] loaded on {DEVICE} as {dtype} in {self._load_seconds:.1f}s")
        return self._model

def _gpu_unload(self):
    with self._model_lock:
        self._model = None
        self._prompt_cache.clear()
        self._load_seconds = 0.0
    import gc
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()

VoiceEngine.load = _gpu_load
VoiceEngine.unload = _gpu_unload
report.append(f"omnivoice  -> {DEVICE} / {'float16' if CUDA else 'float32'} (int8 quantisation off)")

# --------------------------------------------------------------------------- #
# 3. faster-whisper — app/shorts_pipeline.py builds WhisperModel(device="cpu",
#    compute_type="int8"). Wrapping the class forces CUDA + float16 and falls
#    back cleanly when the CUDA libraries are missing.
# --------------------------------------------------------------------------- #
WHISPER_DEVICE, WHISPER_COMPUTE = ("cuda", "float16") if CUDA else ("cpu", "int8")
try:
    import faster_whisper

    _OrigWhisper = faster_whisper.WhisperModel

    class _GpuWhisperModel(_OrigWhisper):
        def __init__(self, model_size_or_path, device="cpu", compute_type="int8", **kwargs):
            try:
                super().__init__(model_size_or_path, device=WHISPER_DEVICE,
                                 compute_type=WHISPER_COMPUTE, **kwargs)
            except Exception as error:
                print(f"  [whisper] GPU load failed ({error}); using CPU int8.")
                super().__init__(model_size_or_path, device="cpu", compute_type="int8", **kwargs)

    faster_whisper.WhisperModel = _GpuWhisperModel
    report.append(f"whisper    -> {WHISPER_DEVICE} / {WHISPER_COMPUTE}")
except ImportError:
    report.append("whisper    -> NOT INSTALLED (transcription and shorts disabled)")

# --------------------------------------------------------------------------- #
# 4. ffmpeg — app/shorts_media.py renders with libx264. Rewriting the command in
#    the shared runner moves encoding onto NVENC, and retries the original x264
#    command if the hardware encoder refuses the stream.
# --------------------------------------------------------------------------- #
from app import shorts_media

def _nvenc_available():
    if not CUDA:
        return False
    try:
        done = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"],
                              capture_output=True, text=True, timeout=120)
        return "h264_nvenc" in done.stdout
    except Exception:
        return False

NVENC = _nvenc_available()
if NVENC:
    _orig_run = shorts_media._run

    def _to_nvenc(command):
        out, index = [], 0
        while index < len(command):
            token = command[index]
            if token == "libx264":
                out.append("h264_nvenc")
            elif token == "-crf":
                # NVENC has no CRF; constant-quality VBR is the equivalent knob.
                out += ["-rc", "vbr", "-cq", command[index + 1]]
                index += 1
            else:
                out.append(token)
            index += 1
        return out

    def _gpu_run(command, *, what, timeout=1800):
        if "libx264" not in command:
            return _orig_run(command, what=what, timeout=timeout)
        try:
            return _orig_run(_to_nvenc(command), what=f"{what} (nvenc)", timeout=timeout)
        except Exception as error:
            print(f"  [ffmpeg] NVENC failed ({str(error)[:160]}); falling back to libx264.")
            return _orig_run(command, what=what, timeout=timeout)

    shorts_media._run = _gpu_run
    report.append("ffmpeg     -> h264_nvenc (x264 fallback)")
else:
    report.append("ffmpeg     -> libx264 (CPU encode)")

# --------------------------------------------------------------------------- #
from app.config import get_settings
from app.services import job_store, voice_store

settings = get_settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("/content/xovato_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device map")
for line in report:
    print("  " + line)
print(f"\n  models dir : {APP_DIR / 'models'}")
print(f"  outputs    : {OUTPUT_DIR}")
print("\nPatched. Run cell ⑤ to open the Feature Studio.")

In [ ]:
#@title ⑤ Feature Studio — pick a feature, set options, press Run { display-mode: "form" }
#@markdown Every feature below calls the same backend functions the FastAPI app calls.
#@markdown Large files: drop them into `/content` with the Files panel on the left and paste the path
#@markdown instead of using the Upload button — the browser upload widget is slow above ~50 MB.

import io, sys, time, traceback, uuid
from pathlib import Path

import ipywidgets as W
from IPython.display import Audio, HTML, Image as IPyImage, Video, clear_output, display

APP_DIR = Path("/content/Xovato-cpu-ai-studio")
sys.path.insert(0, "/content")
import xovato_bootstrap  # app.services shim written by cell 2; see its docstring

xovato_bootstrap.install(APP_DIR)

OUTPUT_DIR = Path("/content/xovato_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from app.config import get_settings
from app.model_manager import model_status
from app.voice_engine import voice_engine
from app.voice_library import STOCK_VOICES, get_stock_voice
from app.voice_styles import EMOTION_PRESETS, instruct_string, resolve_style

settings = get_settings()

# --------------------------------------------------------------------------- #
# Small helpers shared by every panel
# --------------------------------------------------------------------------- #
WIDE = W.Layout(width="460px")


def picker(accept, label, hint=""):
    """Upload button plus a path box, because big media should not go through the browser."""
    upload = W.FileUpload(accept=accept, multiple=False, description="Upload")
    path = W.Text(placeholder="…or a path on this runtime, e.g. /content/clip.wav", layout=WIDE)
    box = W.VBox([W.HTML(f"<b>{label}</b>" + (f" <span style='color:#888'>{hint}</span>" if hint else "")),
                  W.HBox([upload, path])])
    return upload, path, box


def read_picker(upload, path_widget, what="file"):
    """Return (bytes, filename) from whichever input the user actually used."""
    value = upload.value
    if value:
        if isinstance(value, dict):          # ipywidgets 7 (what Colab ships)
            name, item = next(iter(value.items()))
            return bytes(item["content"]), item.get("metadata", {}).get("name", name)
        item = value[0]                      # ipywidgets 8
        return bytes(item["content"]), item["name"]

    text = (path_widget.value or "").strip().strip('"')
    if text:
        source = Path(text)
        if not source.exists():
            raise FileNotFoundError(f"There is no file at {source}")
        return source.read_bytes(), source.name

    raise ValueError(f"Pick a {what} first — upload one, or paste a path.")


def save(name, data: bytes) -> Path:
    destination = OUTPUT_DIR / f"{time.strftime('%H%M%S')}_{name}"
    destination.write_bytes(data)
    return destination


def download_link(path: Path) -> HTML:
    return HTML(f"<code>{path}</code> &nbsp; "
                f"<i>(right-click → Download in the Files panel, or run "
                f"<code>from google.colab import files; files.download('{path}')</code>)</i>")


def show_analysis(analysis):
    print("\n" + analysis.summary)
    for action in analysis.actions:
        print("  •", action)


def timed(label):
    started = time.time()
    print(f"{label} …")
    return lambda: print(f"  finished in {time.time() - started:.1f}s")


# --------------------------------------------------------------------------- #
# Panel: text to speech
# --------------------------------------------------------------------------- #
tts_text = W.Textarea(value="Hello — this voice is running on a Colab GPU.",
                      layout=W.Layout(width="98%", height="90px"), description="Text")
tts_voice = W.Dropdown(options=[("— no preset voice —", "")] + [(f"{v.name} · {v.tagline}", v.id) for v in STOCK_VOICES],
                       value="aria", description="Voice", layout=WIDE)
tts_emotion = W.Dropdown(options=sorted(EMOTION_PRESETS), value="neutral", description="Emotion")
tts_language = W.Text(value="en", description="Language")
tts_preset_pace = W.Checkbox(value=True, description="Use the voice/emotion's own pacing")
tts_speed = W.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.02, description="Speed")
tts_steps = W.IntSlider(value=32, min=4, max=64, step=2, description="Steps",
                        style={"description_width": "initial"})
tts_guidance = W.FloatSlider(value=2.0, min=1.0, max=4.0, step=0.1, description="Guidance")

tts_panel = W.VBox([
    tts_text,
    W.HBox([tts_voice, tts_emotion]),
    W.HBox([tts_language, tts_preset_pace, tts_speed]),
    W.HBox([tts_steps, tts_guidance]),
    W.HTML("<span style='color:#888'>More steps = better quality, slower. On CPU the app used 8; "
           "a GPU can afford 32.</span>"),
])


def run_tts():
    import soundfile as sf

    voice = get_stock_voice(tts_voice.value or None)
    style = resolve_style(
        tts_emotion.value,
        base_instruct=list(voice.instruct_items) if voice else None,
        speed=None if tts_preset_pace.value else tts_speed.value,
        speed_multiplier=voice.default_speed if voice else 1.0,
    )
    print(f"Voice    : {voice.name if voice else 'model default'}")
    print(f"Style    : {style.emotion} | instruct: {instruct_string(style) or 'none'} | speed {style.speed:.2f}")

    stop = timed("Synthesising")
    result = voice_engine.synthesize(
        settings,
        text=tts_text.value,
        style=style,
        language=(tts_language.value or "").strip() or None,
        num_step=tts_steps.value,
        guidance_scale=tts_guidance.value,
    )
    stop()

    destination = OUTPUT_DIR / f"{time.strftime('%H%M%S')}_speech.wav"
    sf.write(str(destination), result.audio, result.sample_rate, subtype="PCM_16")
    seconds = len(result.audio) / max(result.sample_rate, 1)
    print(f"  {seconds:.1f}s of audio, realtime factor {result.generate_seconds / max(seconds, 1e-6):.2f}x")
    display(Audio(str(destination)), download_link(destination))


# --------------------------------------------------------------------------- #
# Panel: voice cloning
# --------------------------------------------------------------------------- #
clone_upload, clone_path, clone_box = picker(
    ".wav,.mp3,.flac,.ogg,.aiff", "Reference clip", "10–30s of clean speech. WAV/MP3/FLAC/OGG (not M4A).")
clone_ref_text = W.Textarea(placeholder="The exact words spoken in the reference clip — required",
                            layout=W.Layout(width="98%", height="60px"), description="Transcript")
clone_text = W.Textarea(value="This is my cloned voice reading a new sentence.",
                        layout=W.Layout(width="98%", height="80px"), description="Say this")
clone_language = W.Text(value="en", description="Language")
clone_emotion = W.Dropdown(options=sorted(EMOTION_PRESETS), value="neutral", description="Emotion")
clone_steps = W.IntSlider(value=32, min=4, max=64, step=2, description="Steps")

clone_panel = W.VBox([
    clone_box, clone_ref_text, clone_text,
    W.HBox([clone_language, clone_emotion, clone_steps]),
    W.HTML("<span style='color:#888'>The transcript must match the clip. A wrong one measurably "
           "degrades the clone, and an empty one makes OmniVoice try to download an ASR model.</span>"),
])


def run_clone():
    import soundfile as sf

    data, name = read_picker(clone_upload, clone_path, "reference clip")
    if not clone_ref_text.value.strip():
        raise ValueError("Add the transcript of the reference clip — cloning needs it.")

    reference = OUTPUT_DIR / f"reference_{Path(name).stem}{Path(name).suffix or '.wav'}"
    reference.write_bytes(data)
    print(f"Reference: {reference.name} ({len(data)/1e6:.1f} MB)")

    style = resolve_style(clone_emotion.value)
    stop = timed("Cloning and synthesising")
    result = voice_engine.synthesize(
        settings,
        text=clone_text.value,
        style=style,
        language=(clone_language.value or "").strip() or None,
        ref_audio=reference,
        ref_text=clone_ref_text.value.strip(),
        cache_key=reference.stem,
        num_step=clone_steps.value,
    )
    stop()

    destination = OUTPUT_DIR / f"{time.strftime('%H%M%S')}_cloned.wav"
    sf.write(str(destination), result.audio, result.sample_rate, subtype="PCM_16")
    print(f"  cloned={result.cloned}")
    display(HTML("<b>Reference</b>"), Audio(str(reference)),
            HTML("<b>Cloned result</b>"), Audio(str(destination)), download_link(destination))


# --------------------------------------------------------------------------- #
# Panel: transcription
# --------------------------------------------------------------------------- #
stt_upload, stt_path, stt_box = picker("audio/*,video/*", "Audio or video", "any format ffmpeg reads")
stt_model = W.Dropdown(options=["tiny", "base", "small", "medium", "large-v3"], value="base", description="Model")
stt_srt = W.Checkbox(value=True, description="Also write an .srt subtitle file")

stt_panel = W.VBox([stt_box, W.HBox([stt_model, stt_srt]),
                    W.HTML("<span style='color:#888'>Weights download on first use: tiny ≈75 MB, "
                           "large-v3 ≈3 GB.</span>")])


def _srt_time(seconds):
    ms = int(round(seconds * 1000))
    hours, ms = divmod(ms, 3_600_000)
    minutes, ms = divmod(ms, 60_000)
    secs, ms = divmod(ms, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"


def run_stt():
    from app.shorts_media import extract_audio, probe_media
    from app.shorts_pipeline import transcribe_audio

    data, name = read_picker(stt_upload, stt_path, "media file")
    source = OUTPUT_DIR / f"stt_{Path(name).name}"
    source.write_bytes(data)

    probe = probe_media(source)
    print(f"Input    : {name} | {probe.duration_sec or 0:.1f}s | audio={probe.has_audio} video={probe.has_video}")
    if not probe.has_audio:
        raise ValueError("This file has no audio track.")

    audio = extract_audio(source, OUTPUT_DIR) if probe.has_video else source
    stop = timed(f"Transcribing with faster-whisper '{stt_model.value}'")
    transcript = transcribe_audio(audio, provider="local", model_size=stt_model.value)
    stop()

    print(f"  engine {transcript.engine} | language {transcript.language} | "
          f"{len(transcript.segments)} segments, {len(transcript.words)} words\n")
    text_path = OUTPUT_DIR / f"{source.stem}.txt"
    text_path.write_text(transcript.full_text, encoding="utf-8")

    for segment in transcript.segments[:40]:
        print(f"  [{segment.start:7.2f} → {segment.end:7.2f}]  {segment.text}")
    if len(transcript.segments) > 40:
        print(f"  … {len(transcript.segments) - 40} more segments in the .txt")

    display(download_link(text_path))
    if stt_srt.value:
        lines = []
        for index, segment in enumerate(transcript.segments, start=1):
            lines.append(f"{index}\n{_srt_time(segment.start)} --> {_srt_time(segment.end)}\n{segment.text}\n")
        srt_path = OUTPUT_DIR / f"{source.stem}.srt"
        srt_path.write_text("\n".join(lines), encoding="utf-8")
        display(download_link(srt_path))


# --------------------------------------------------------------------------- #
# Panels: images
# --------------------------------------------------------------------------- #
img_upload, img_path, img_box = picker("image/*", "Image", "JPEG / PNG / WEBP")
img_resolution = W.Dropdown(options=["source", "1080p", "2k", "4k"], value="source", description="Resolution")
img_format = W.Dropdown(options=["jpg", "png", "webp"], value="jpg", description="Format")
img_quality = W.IntSlider(value=92, min=60, max=100, description="Quality")
img_strength = W.FloatSlider(value=1.0, min=0.0, max=1.5, step=0.05, description="Strength")

image_panel = W.VBox([img_box, W.HBox([img_resolution, img_format]),
                      W.HBox([img_quality, img_strength]),
                      W.HTML("<span style='color:#888'>Corrections are chosen from the image's own "
                             "histogram — exposure, contrast, colour cast, detail.</span>")])

bg_upload, bg_path, bg_box = picker("image/*", "Image", "the subject is kept, the background becomes alpha")
bg_softness = W.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="Edge softness")
bg_resolution = W.Dropdown(options=["source", "1080p", "2k", "4k"], value="source", description="Resolution")
bg_panel = W.VBox([bg_box, W.HBox([bg_resolution, bg_softness]),
                   W.HTML("<span style='color:#888'>Output is PNG with real per-pixel alpha. "
                          "Needs <code>models/segmentation/u2netp.onnx</code>.</span>")])

face_upload, face_path, face_box = picker("image/*", "Photo with faces")
face_blend = W.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="Blend")
face_whole = W.Checkbox(value=True, description="Also correct the rest of the photo")
face_resolution = W.Dropdown(options=["source", "1080p", "2k", "4k"], value="source", description="Resolution")
face_panel = W.VBox([face_box, W.HBox([face_resolution, face_blend]), face_whole,
                     W.HTML("<span style='color:#888'>YuNet finds the faces, GPEN-BFR-256 rebuilds them. "
                            "Needs <code>models/face/yunet.onnx</code> and "
                            "<code>models/face/gpen_bfr_256.onnx</code>.</span>")])


def _show_before_after(original: bytes, result_path: Path, analysis):
    display(HTML("<b>Before</b>"), IPyImage(data=original, width=380))
    display(HTML("<b>After</b>"), IPyImage(filename=str(result_path), width=380))
    show_analysis(analysis)
    display(download_link(result_path))


def run_image_enhance():
    from app.image_processing import enhance_image

    data, name = read_picker(img_upload, img_path, "image")
    stop = timed("Enhancing")
    payload, analysis = enhance_image(
        data,
        resolution=img_resolution.value,
        output_format=img_format.value,
        quality=img_quality.value,
        strength=img_strength.value,
    )
    stop()
    _show_before_after(data, save(f"enhanced_{Path(name).stem}.{img_format.value}", payload), analysis)


def run_background_removal():
    from app.image_processing import remove_background

    data, name = read_picker(bg_upload, bg_path, "image")
    if not settings.segmentation_model_path.exists():
        raise FileNotFoundError(f"u2netp.onnx is missing at {settings.segmentation_model_path} — rerun cell 3.")
    stop = timed("Segmenting")
    payload, analysis = remove_background(
        data,
        model_path=settings.segmentation_model_path,
        resolution=bg_resolution.value,
        output_format="png",
        edge_softness=bg_softness.value,
    )
    stop()
    _show_before_after(data, save(f"cutout_{Path(name).stem}.png", payload), analysis)


def run_face_restoration():
    from app.image_processing import restore_faces_image

    data, name = read_picker(face_upload, face_path, "image")
    for path in (settings.face_detector_path, settings.face_restorer_path):
        if not path.exists():
            raise FileNotFoundError(f"{path.name} is missing at {path} — rerun cell 3 with GET_FACE ticked.")
    stop = timed("Restoring faces")
    payload, analysis = restore_faces_image(
        data,
        detector_path=settings.face_detector_path,
        restorer_path=settings.face_restorer_path,
        resolution=face_resolution.value,
        blend=face_blend.value,
        enhance_whole_image=face_whole.value,
    )
    stop()
    print(f"Faces detected: {analysis.faces_detected}")
    _show_before_after(data, save(f"faces_{Path(name).stem}.jpg", payload), analysis)


# --------------------------------------------------------------------------- #
# Panel: audio enhancement
# --------------------------------------------------------------------------- #
aud_upload, aud_path, aud_box = picker("audio/*", "Recording", "WAV / FLAC / OGG")
aud_denoise = W.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05, description="Denoise")
aud_lufs = W.FloatSlider(value=-16.0, min=-28.0, max=-8.0, step=0.5, description="Target LUFS")
aud_ceiling = W.FloatSlider(value=-1.0, min=-6.0, max=0.0, step=0.1, description="Peak dBFS")
aud_highpass = W.FloatSlider(value=80.0, min=0.0, max=200.0, step=5.0, description="Highpass Hz")
aud_format = W.Dropdown(options=["wav", "flac", "ogg"], value="wav", description="Format")

audio_panel = W.VBox([aud_box, W.HBox([aud_denoise, aud_lufs]), W.HBox([aud_ceiling, aud_highpass]), aud_format,
                      W.HTML("<span style='color:#888'>Spectral noise gate → loudness normalisation → "
                             "true-peak limiter, in that order.</span>")])


def run_audio_enhance():
    from app.audio_processing import enhance_audio

    data, name = read_picker(aud_upload, aud_path, "audio file")
    stop = timed("Cleaning up")
    payload, resolved, analysis = enhance_audio(
        data,
        denoise_strength=aud_denoise.value,
        target_lufs=aud_lufs.value,
        peak_ceiling_dbfs=aud_ceiling.value,
        highpass_hz=aud_highpass.value,
        output_format=aud_format.value,
    )
    stop()
    destination = save(f"clean_{Path(name).stem}.{resolved}", payload)
    display(HTML("<b>Before</b>"), Audio(data=data), HTML("<b>After</b>"), Audio(filename=str(destination)))
    show_analysis(analysis)
    display(download_link(destination))


# --------------------------------------------------------------------------- #
# Panel: shorts
# --------------------------------------------------------------------------- #
shorts_upload, shorts_path, shorts_box = picker(
    "video/*,audio/*", "Long video", "paste a /content path for anything over ~50 MB")
shorts_count = W.IntSlider(value=3, min=1, max=10, description="Clips")
shorts_min = W.FloatSlider(value=12.0, min=3.0, max=60.0, step=1.0, description="Min sec")
shorts_max = W.FloatSlider(value=45.0, min=10.0, max=180.0, step=1.0, description="Max sec")
shorts_captions = W.Checkbox(value=True, description="Burn in captions")
shorts_render_area = W.Output()

shorts_panel = W.VBox([shorts_box, W.HBox([shorts_count, shorts_min, shorts_max]), shorts_captions,
                       W.HTML("<span style='color:#888'>Transcribe → rank the strongest moments → "
                              "crop to 9:16 and render. Set <code>OPENAI_API_KEY</code> in the environment "
                              "to rank with an LLM instead of the heuristic.</span>"),
                       shorts_render_area])


def run_shorts():
    from app.models import JobCreate, MediaType
    from app.services import job_store
    from app.shorts_service import create_analysis_job, project_dir, run_analysis, run_render, save_upload

    data, name = read_picker(shorts_upload, shorts_path, "video")
    project_id = f"colab_{uuid.uuid4().hex[:8]}"
    source = save_upload(settings, project_id, name, io.BytesIO(data))
    print(f"Project  : {project_id}\nSource   : {source} ({source.stat().st_size/1e6:.1f} MB)")

    job_id = create_analysis_job(project_id, name)
    stop = timed("Analysing (probe → extract audio → transcribe → rank)")
    run_analysis(
        settings,
        job_id=job_id,
        project_id=project_id,
        source=source,
        target_count=shorts_count.value,
        min_len=shorts_min.value,
        max_len=shorts_max.value,
        transcribe_provider="local",
        ranking_provider=None,
    )
    stop()

    job = job_store.get(job_id)
    if job.status != "completed":
        raise RuntimeError(job.error or "Analysis failed.")

    candidates = job.result.get("candidates", [])
    print(f"  ranked by {job.result.get('ranking_engine')} from {job.result.get('transcribe_engine')}\n")
    boxes = []
    for candidate in candidates:
        label = (f"{candidate['start_sec']:.1f}s → {candidate['end_sec']:.1f}s "
                 f"(score {candidate.get('score', 0):.2f})  {candidate.get('hook', '')[:80]}")
        boxes.append(W.Checkbox(value=True, description=label, indent=False,
                                layout=W.Layout(width="98%"), style={"description_width": "initial"}))

    render_button = W.Button(description=f"Render selected to 9:16", button_style="success", icon="film")

    def on_render(_):
        with shorts_render_area:
            clear_output()
            chosen = [candidate for candidate, box in zip(candidates, boxes) if box.value]
            if not chosen:
                print("Nothing selected.")
                return
            render_job = job_store.create(JobCreate(
                module="shorts-studio", media_type=MediaType.video,
                input_uri=f"local://shorts/{project_id}", model_chain=["ffmpeg-render"],
                settings={"project_id": project_id},
            ))
            stop_render = timed(f"Rendering {len(chosen)} clip(s)")
            run_render(settings, job_id=render_job.id, project_id=project_id, source=source,
                       clips=chosen, captions=shorts_captions.value)
            stop_render()

            finished = job_store.get(render_job.id)
            if finished.error:
                print("Render error:", finished.error)
            for clip in finished.result.get("clips", []):
                path = project_dir(settings, project_id) / Path(clip["uri"]).name
                print(f"\nClip {clip['index']}: {clip['start_sec']}s → {clip['end_sec']}s "
                      f"({clip['size_bytes']/1e6:.1f} MB)  {clip['hook'][:70]}")
                if path.exists() and path.stat().st_size < 40_000_000:
                    display(Video(str(path), embed=True, width=260))
                display(download_link(path))

    render_button.on_click(on_render)
    with shorts_render_area:
        clear_output()
        display(W.VBox(boxes + [render_button]))


# --------------------------------------------------------------------------- #
# Panel: status
# --------------------------------------------------------------------------- #
status_panel = W.VBox([W.HTML("<span style='color:#888'>Everything the runtime currently has loaded "
                              "and every model path the app looks for.</span>")])


def run_status():
    import subprocess

    import torch

    print("Runtime")
    print(f"  torch {torch.__version__} | cuda {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"  {torch.cuda.get_device_name(0)} | {free/1e9:.1f} GB free of {total/1e9:.1f} GB")
    try:
        import onnxruntime as ort
        print(f"  onnxruntime {ort.__version__} | providers: {', '.join(ort.get_available_providers())}")
    except ImportError:
        print("  onnxruntime not installed")

    print("\nModels")
    for row in model_status():
        print(f"  {'PRESENT' if row['exists'] else 'missing':>8}  {row['id']:<20} {row['feature']}")
        print(f"            {row['path']}")

    print("\nVoice engine")
    for key, value in voice_engine.status().items():
        print(f"  {key}: {value}")

    print("\nOutputs")
    files = sorted(OUTPUT_DIR.glob("*"))
    if not files:
        print("  (nothing yet)")
    for path in files[-25:]:
        print(f"  {path.stat().st_size/1e6:8.2f} MB  {path}")

    print("\nGPU")
    print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout or "  no GPU")


# --------------------------------------------------------------------------- #
# The picker itself
# --------------------------------------------------------------------------- #
FEATURES = {
    "Text → Speech": (tts_panel, run_tts),
    "Voice Cloning": (clone_panel, run_clone),
    "Speech → Text (+ SRT)": (stt_panel, run_stt),
    "Image Enhancement": (image_panel, run_image_enhance),
    "Background Removal": (bg_panel, run_background_removal),
    "Face Restoration": (face_panel, run_face_restoration),
    "Audio Enhancement": (audio_panel, run_audio_enhance),
    "Shorts (vertical clips)": (shorts_panel, run_shorts),
    "System & Model Status": (status_panel, run_status),
}

feature_select = W.Dropdown(options=list(FEATURES), value="Text → Speech", description="Feature",
                            layout=W.Layout(width="420px"), style={"description_width": "initial"})
run_button = W.Button(description="Run", button_style="primary", icon="play", layout=W.Layout(width="140px"))
options_area = W.VBox([])
result_area = W.Output(layout=W.Layout(border="1px solid #ddd", padding="8px", min_height="80px"))


def on_feature_change(_=None):
    options_area.children = [FEATURES[feature_select.value][0]]
    run_button.description = f"Run · {feature_select.value.split(' (')[0]}"
    with result_area:
        clear_output()


def on_run(_):
    with result_area:
        clear_output()
        run_button.disabled = True
        try:
            FEATURES[feature_select.value][1]()
        except Exception as error:
            print(f"{type(error).__name__}: {error}\n")
            traceback.print_exc(limit=4)
        finally:
            run_button.disabled = False


feature_select.observe(on_feature_change, names="value")
run_button.on_click(on_run)
on_feature_change()

display(W.VBox([
    W.HTML("<h3 style='margin-bottom:4px'>Xovato Feature Studio</h3>"
           "<div style='color:#888;margin-bottom:8px'>Pick a feature — the options below change to match it.</div>"),
    W.HBox([feature_select, run_button]),
    options_area,
    W.HTML("<b>Result</b>"),
    result_area,
]))

---

## Optional — run the real REST API

Everything above calls the backend functions directly. If you want the actual FastAPI service
(the same one the studio frontend talks to), run the cell below. It starts **in this kernel**, so it
inherits the GPU patches from cell ④ — starting `uvicorn` in a subprocess would not.

Endpoints include `POST /api/v1/voice/speech`, `POST /api/v1/images/enhance`,
`POST /api/v1/images/background-removal`, `POST /api/v1/images/face-restoration`,
`POST /api/v1/audio/enhance`, `POST /api/v1/shorts/analyze`, plus the OpenAI-compatible
`POST /v1/audio/speech`. Interactive docs live at `/docs`.

In [ ]:
#@title ⑥ (optional) Serve the FastAPI backend on a public Colab URL { display-mode: "form" }
PORT = 8000 #@param {type:"integer"}

import sys, threading, time

sys.path.insert(0, "/content")
import xovato_bootstrap  # app.services shim; app/main.py needs it too

xovato_bootstrap.install()

import uvicorn

from app.main import app as fastapi_app

config = uvicorn.Config(fastapi_app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
# Signal handlers can only be installed on the main thread.
server.install_signal_handlers = lambda: None

threading.Thread(target=server.run, daemon=True).start()
for _ in range(60):
    if server.started:
        break
    time.sleep(0.5)

print("started:", server.started)

try:
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({PORT})").rstrip("/")
    print(f"\nAPI    : {url}/api/v1/health")
    print(f"Docs   : {url}/docs")
    print("\nThe URL is private to your Colab session and dies with the runtime.")
except ImportError:
    print(f"\nNot running in Colab — the API is on http://localhost:{PORT}/docs")

import httpx

print("\nHealth check:")
print(httpx.get(f"http://127.0.0.1:{PORT}/api/v1/health", timeout=30).json())

In [ ]:
#@title ⑦ (optional) Housekeeping — download everything, free VRAM { display-mode: "form" }
ZIP_AND_DOWNLOAD = True  #@param {type:"boolean"}
UNLOAD_TTS_MODEL = False #@param {type:"boolean"}

import shutil
from pathlib import Path

OUTPUT_DIR = Path("/content/xovato_outputs")
files = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file())
print(f"{len(files)} file(s) in {OUTPUT_DIR}, {sum(p.stat().st_size for p in files)/1e6:.1f} MB total")

if ZIP_AND_DOWNLOAD and files:
    archive = shutil.make_archive("/content/xovato_outputs", "zip", str(OUTPUT_DIR))
    print("archive:", archive)
    try:
        from google.colab import files as colab_files

        colab_files.download(archive)
    except ImportError:
        pass

if UNLOAD_TTS_MODEL:
    from app.voice_engine import voice_engine

    voice_engine.unload()
    import torch

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM after unload: {free/1e9:.1f} GB free of {total/1e9:.1f} GB")